# META-CXR — encoder inference sensitivity

This notebook does **not** train seven models. It loads the single validation-selected E123 checkpoint, runs BioViL-T + PubMedCLIP + SwinV2 once per held-out batch, and evaluates seven downstream token masks. The deltas are descriptive sensitivity measurements, not causal contributions.

In [ ]:
DATASET_SLUG = "phuong20052/mimic-cxr-jpg-dataset"
CHECKPOINT_INPUT_SLUG = "phuong20052/meta-cxr-checkpoints"
CHECKPOINT_DATASET_HANDLE = "phuong20052/meta-cxr-checkpoints"       # private owner/slug matching the attached checkpoint
RESULT_DATASET_HANDLE = ""           # pre-created private owner/slug
REPO_COMMIT = "b3e10f480febda49d0d2ad6e01d6ae2ec86a241e"                     # exact 40-character smoke-repo commit
BATCH_SIZE = 2
NUM_WORKERS = 4
SEED = 42


In [ ]:
import os, pathlib, subprocess, sys
for name, value in {'DATASET_SLUG': DATASET_SLUG, 'CHECKPOINT_INPUT_SLUG': CHECKPOINT_INPUT_SLUG, 'CHECKPOINT_DATASET_HANDLE': CHECKPOINT_DATASET_HANDLE, 'RESULT_DATASET_HANDLE': RESULT_DATASET_HANDLE}.items():
    if not value:
        raise ValueError(f'{name} is required')
if len(REPO_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in REPO_COMMIT.lower()):
    raise ValueError('REPO_COMMIT must be an exact 40-character SHA')
repo_dir = pathlib.Path('/kaggle/working/META-CXR-SMOKETEST')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/minhphuong150505/META-CXR-SMOKETEST.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth=1', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', REPO_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip() != REPO_COMMIT:
    raise RuntimeError('Exact commit checkout failed')
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))


In [ ]:
# Checkpoints contain non-weight provenance and RNG state.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'


In [ ]:
from smoke.runtime import load_kaggle_secrets
load_kaggle_secrets(('GCS_SERVICE_ACCOUNT', 'WANDB_API_KEY', 'HF_TOKEN', 'KAGGLE_API_TOKEN'), '/kaggle/working/.meta-cxr-secrets')
print('Loaded required secrets into OS environment (values hidden).')


In [ ]:
import json
from smoke.runtime import environment_fingerprint, assert_two_t4, compatibility_matrix
before = environment_fingerprint()
print(json.dumps(before, indent=2, sort_keys=True))
assert_two_t4(before)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', 'hi-ml-multimodal==0.2.1'], check=True)
after = environment_fingerprint()
assert_two_t4(after)
print(json.dumps(compatibility_matrix(before, after), indent=2, sort_keys=True))


In [ ]:
from smoke.runtime import discover_dataset, load_dataset_manifest, write_runtime_env_config
from smoke.checkpoints import assert_private_kaggle_dataset
assert_private_kaggle_dataset(CHECKPOINT_DATASET_HANDLE)
dataset_root = discover_dataset(DATASET_SLUG)
dataset_manifest, _, dataset_hash = load_dataset_manifest(dataset_root)
if dataset_manifest.get('status') != 'qa_passed':
    raise RuntimeError('Dataset manifest is not QA-passed')
write_runtime_env_config(dataset_root, '/kaggle/working/meta-cxr-sensitivity')
checkpoint_root = pathlib.Path('/kaggle/input') / CHECKPOINT_INPUT_SLUG.split('/')[-1]
checkpoint = checkpoint_root / 'checkpoint_best.pth'
if not checkpoint.is_file():
    raise FileNotFoundError('Attached private checkpoint_best.pth is missing')


In [ ]:
import torch
checkpoint_meta = torch.load(checkpoint, map_location='cpu')
identity = checkpoint_meta.get('identity')
if not identity or identity.get('source_commit') != REPO_COMMIT or identity.get('dataset_manifest_sha256') != dataset_hash:
    raise RuntimeError('Checkpoint/source/dataset identity mismatch')
result_dir = pathlib.Path('/kaggle/working/meta-cxr-sensitivity-results')
result_path = result_dir / 'encoder_sensitivity.json'
subprocess.run([sys.executable, 'scripts/evaluate_encoder_sensitivity.py', '--cfg-path', 'pretraining/configs/stage1_smoke_2xt4.yaml', '--checkpoint', str(checkpoint), '--output', str(result_path), '--source-commit', REPO_COMMIT, '--dataset-manifest-sha256', dataset_hash, '--config-fingerprint', identity['config_fingerprint'], '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS)], check=True)
summary = json.loads(result_path.read_text())
print({'status': summary['status'], 'method': summary['method'], 'test_studies': summary['test_studies'], 'wall_seconds': summary['wall_seconds'], 'peak_vram_bytes': summary['peak_vram_bytes']})
for run_id, report in summary['reports'].items():
    print(run_id, report['aggregates']['positive_macro_f1'])


In [ ]:
from smoke.checkpoints import upload_private_results_dataset
upload_private_results_dataset(RESULT_DATASET_HANDLE, result_dir)
print('Private aggregate result upload and manifest verification succeeded; local files retained.')
